# Week 3 Day 5 — GPU Capacity Benchmark

**Completed reference Colab notebook**

> ⚠️ **Important / مهم:** The saved benchmark numbers in this notebook are **illustrative simulated reference data**, not proof of a real T4 run. To produce valid measured results, set `USE_REFERENCE_DATA = False`, select a **T4 GPU** in Colab, and run all cells. The notebook will replace the reference report with live measurements.

Goal: serve the locked model with vLLM, sweep concurrency `1,2,4,8,16`, find the highest level meeting the p95 latency SLO, write the capacity note, and run the green check.


## 0. Prediction card

- Predicted knee concurrency: **8**
- Target p95 end-to-end latency: **2.5 seconds**
- Locked model: **Qwen/Qwen2.5-1.5B-Instruct**
- Quantisation: **none (FP16 serving path)**
- Tool-call parser: **hermes**


In [1]:
from pathlib import Path
import json, os, signal, subprocess, sys, time

# Keep True only to inspect the completed example without a GPU.
# Set False in Colab to collect real T4 measurements.
USE_REFERENCE_DATA = True

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
QUANTISATION = "none"
TOOL_CALL_PARSER = "hermes"
TARGET_P95_S = 2.5
PORT = 8000
WORKDIR = Path("/content") if Path("/content").exists() else Path.cwd()
os.chdir(WORKDIR)

print("mode:", "SIMULATED REFERENCE" if USE_REFERENCE_DATA else "LIVE T4 MEASUREMENT")
print("model:", MODEL)
print("quantisation:", QUANTISATION)
print("tool-call parser:", TOOL_CALL_PARSER)
print("target p95:", TARGET_P95_S, "seconds")
print("workdir:", WORKDIR)


mode: SIMULATED REFERENCE
model: Qwen/Qwen2.5-1.5B-Instruct
quantisation: none
tool-call parser: hermes
target p95: 2.5 seconds
workdir: /content


## 1. Runtime and dependencies

For a real run, choose **Runtime → Change runtime type → T4 GPU** before running this section.


In [2]:
if USE_REFERENCE_DATA:
    print("Reference mode: hardware probe skipped. A live run executes nvidia-smi.")
else:
    subprocess.run(["nvidia-smi"], check=True)


Reference mode: hardware probe skipped. A live run executes nvidia-smi.


In [3]:
VLLM_PIN = "0.6.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

if USE_REFERENCE_DATA:
    print("Reference mode: package installation skipped.")
else:
    pip_install(
        f"vllm=={VLLM_PIN}",
        f"transformers=={TRANSFORMERS_PIN}",
        f"accelerate=={ACCELERATE_PIN}",
        f"httpx=={HTTPX_PIN}",
        f"openai=={OPENAI_PIN}",
    )
    print("serving pins installed")


Reference mode: package installation skipped.


## 2. Prepare the prompts, benchmark harness, and verifier

The supplied ZIP did not contain `bench.py`, so this notebook creates a self-contained async streaming harness with the required report schema.


In [4]:
PROMPTS = "What is a GPU?\nDefine tokens per second in one line.\nExplain the difference between prefill and decode in two sentences.\nList three reasons decode is memory-bound rather than compute-bound.\nSummarise what an inference server does for an ops team, in three short bullets.\nWhy does a longer prompt increase time to first token but not the per-token gap?\nDescribe the KV cache to a new engineer and say why it grows with context length.\nWalk through what continuous batching changes versus static batching, with an example of the straggler effect it removes.\nName two things weight-only quantisation trades away in exchange for smaller memory footprint.\nA user asks for the weather in Riyadh and the current time in Tokyo; describe the two tool calls you would make and the arguments for each.\nWrite a short runbook for rolling back a bad deployment, listing the steps in order and the check after each one.\nExplain, for a non-technical manager, why a busy GPU is not the same as a productive GPU, using the utilisation trap.\nCompare fp16 and int4 for serving a 1.5 billion parameter model: memory, speed, and quality, in a short paragraph each.\nGive a one-sentence definition of p95 latency and say why it matters more than the average for an SLO.\nDraft three sentences a platform team could send another team to describe an OpenAI-compatible endpoint they can call.\nOutline the symptom, hypothesis, and measurement steps you would take when throughput is lower than expected under load.\nWhat is PagedAttention and what problem in KV cache memory does it solve? Answer in two sentences.\nExplain why the knee at the SLO, not the peak throughput, is the honest capacity number for a benchmark.\nDescribe how you would size the GPU memory budget for a model plus its KV cache before ever loading it.\nWrite a calm status update for a channel of engineers explaining that latency has risen, what you suspect, and what you are doing about it, in four sentences.\n"
BENCH_SOURCE = "#!/usr/bin/env python3\nimport argparse, asyncio, json, math, time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport httpx\n\ndef pct(values, q):\n    if not values:\n        return 0.0\n    xs = sorted(values)\n    pos = (len(xs) - 1) * q\n    lo, hi = math.floor(pos), math.ceil(pos)\n    if lo == hi:\n        return xs[lo]\n    return xs[lo] * (hi - pos) + xs[hi] * (pos - lo)\n\nasync def one_request(client, model, prompt, semaphore):\n    async with semaphore:\n        started = time.perf_counter()\n        first_token_at = None\n        completion_tokens = 0\n        text_parts = []\n        payload = {\n            \"model\": model,\n            \"messages\": [{\"role\": \"user\", \"content\": prompt}],\n            \"temperature\": 0,\n            \"max_tokens\": 64,\n            \"stream\": True,\n            \"stream_options\": {\"include_usage\": True},\n        }\n        try:\n            async with client.stream(\"POST\", \"/chat/completions\", json=payload) as response:\n                response.raise_for_status()\n                async for line in response.aiter_lines():\n                    if not line.startswith(\"data: \"):\n                        continue\n                    body = line[6:]\n                    if body == \"[DONE]\":\n                        break\n                    chunk = json.loads(body)\n                    choices = chunk.get(\"choices\") or []\n                    if choices:\n                        content = (choices[0].get(\"delta\") or {}).get(\"content\")\n                        if content:\n                            text_parts.append(content)\n                            if first_token_at is None:\n                                first_token_at = time.perf_counter()\n                    usage = chunk.get(\"usage\") or {}\n                    completion_tokens = usage.get(\"completion_tokens\", completion_tokens)\n            finished = time.perf_counter()\n            if first_token_at is None:\n                first_token_at = finished\n            if not completion_tokens:\n                completion_tokens = max(1, len(\"\".join(text_parts).split()))\n            return {\n                \"ok\": True,\n                \"ttft\": first_token_at - started,\n                \"latency\": finished - started,\n                \"tokens\": completion_tokens,\n            }\n        except Exception as exc:\n            return {\"ok\": False, \"error\": f\"{type(exc).__name__}: {exc}\"}\n\nasync def run_level(client, model, prompts, concurrency, count):\n    semaphore = asyncio.Semaphore(concurrency)\n    started = time.perf_counter()\n    tasks = [one_request(client, model, prompts[i % len(prompts)], semaphore) for i in range(count)]\n    results = await asyncio.gather(*tasks)\n    wall = time.perf_counter() - started\n    good = [r for r in results if r[\"ok\"]]\n    return {\n        \"concurrency\": concurrency,\n        \"tokens_per_s\": round(sum(r[\"tokens\"] for r in good) / wall, 3) if wall else 0.0,\n        \"ttft_p50_s\": round(pct([r[\"ttft\"] for r in good], .50), 4),\n        \"ttft_p95_s\": round(pct([r[\"ttft\"] for r in good], .95), 4),\n        \"latency_p95_s\": round(pct([r[\"latency\"] for r in good], .95), 4),\n        \"errors\": len(results) - len(good),\n    }\n\nasync def main(args):\n    prompts = [line.strip() for line in Path(args.prompt_file).read_text().splitlines() if line.strip()]\n    levels = [int(x) for x in args.concurrency.split(\",\")]\n    timeout = httpx.Timeout(180.0)\n    async with httpx.AsyncClient(base_url=args.base_url.rstrip(\"/\") + \"/v1\", timeout=timeout) as client:\n        print(\"warm-up request (excluded)\")\n        warm = await one_request(client, args.model, prompts[0], asyncio.Semaphore(1))\n        if not warm[\"ok\"]:\n            raise RuntimeError(\"warm-up failed: \" + warm[\"error\"])\n        measured = []\n        for c in levels:\n            row = await run_level(client, args.model, prompts, c, args.requests_per_level)\n            measured.append(row)\n            print(f\"c={c:>2} tok/s={row['tokens_per_s']:>7.1f} p95={row['latency_p95_s']:.3f}s errors={row['errors']}\")\n    out = Path(args.out)\n    document = json.loads(out.read_text()) if out.exists() else {\"runs\": []}\n    document.setdefault(\"runs\", []).append({\n        \"timestamp_utc\": datetime.now(timezone.utc).isoformat(),\n        \"model\": args.model,\n        \"requests_per_level\": args.requests_per_level,\n        \"measured\": True,\n        \"levels\": measured,\n    })\n    out.write_text(json.dumps(document, indent=2) + \"\n\")\n    print(\"saved\", out)\n\nif __name__ == \"__main__\":\n    p = argparse.ArgumentParser()\n    p.add_argument(\"--base-url\", required=True)\n    p.add_argument(\"--model\", required=True)\n    p.add_argument(\"--concurrency\", required=True)\n    p.add_argument(\"--requests-per-level\", type=int, default=20)\n    p.add_argument(\"--prompt-file\", required=True)\n    p.add_argument(\"--out\", required=True)\n    asyncio.run(main(p.parse_args()))\n"
VERIFY_SOURCE = "# Green-check verifier for Lab W3D5 (benchmark harness).\n# Paste this as the last cell of your day-5 notebook and run it. It reads\n# bench_report.json (from the harness) and capacity-note.md, and checks the\n# schema, that at least four concurrency levels ran, that errors are zero or\n# explained, and that the capacity note is filled in.\n#\n# Last line is exactly one of:\n#   GREEN CHECK: PASS\n#   GREEN CHECK: FAIL (<reason>)\n# No interactivity, no arguments; exit code matches.\n\nimport json, os, re\n\nLEVEL_KEYS = {\"concurrency\", \"tokens_per_s\", \"ttft_p50_s\", \"ttft_p95_s\",\n              \"latency_p95_s\", \"errors\"}\n\n\nclass _Stop(Exception):\n    \"\"\"Ends the check without killing the notebook kernel.\"\"\"\n\n\ndef fail(reason: str) -> \"NoReturn\":\n    print(f\"GREEN CHECK: FAIL ({reason})\")\n    raise _Stop()\n\n\ndef main() -> None:\n    # 1) bench report\n    if not os.path.exists(\"bench_report.json\"):\n        fail(\"bench_report.json not found; run the harness in Cell 3\")\n    try:\n        with open(\"bench_report.json\") as fh:\n            document = json.load(fh)\n    except json.JSONDecodeError as exc:\n        fail(f\"bench_report.json is not valid JSON: {exc}\")\n\n    # bench.py appends each sweep to a \"runs\" list rather than overwriting, so\n    # the file is a document and the thing to grade is the most recent run. A\n    # bare list is also accepted, for a report assembled by hand.\n    if isinstance(document, dict) and isinstance(document.get(\"runs\"), list):\n        if not document[\"runs\"]:\n            fail(\"bench_report.json has no runs; the harness wrote nothing\")\n        levels = document[\"runs\"][-1].get(\"levels\")\n        if not isinstance(levels, list):\n            fail(\"the most recent run in bench_report.json has no levels list\")\n    elif isinstance(document, list):\n        levels = document\n    else:\n        fail(\"bench_report.json must be the harness output ({'runs': [...]}) \"\n             \"or a bare list of per-level objects\")\n    if len(levels) < 4:\n        fail(f\"need at least 4 concurrency levels, found {len(levels)}\")\n\n    total_errors = 0\n    for i, L in enumerate(levels):\n        if not isinstance(L, dict):\n            fail(f\"level {i} is not an object\")\n        missing = LEVEL_KEYS - set(L)\n        if missing:\n            fail(f\"level {i} missing keys: {sorted(missing)}\")\n        if not isinstance(L[\"errors\"], int) or L[\"errors\"] < 0:\n            fail(f\"level {i} errors must be a non-negative integer\")\n        total_errors += L[\"errors\"]\n\n    # 2) the knee file from Cell 5\n    if not os.path.exists(\"knee.json\"):\n        fail(\"knee.json not found; write it in Cell 5\")\n    try:\n        with open(\"knee.json\") as fh:\n            knee = json.load(fh)\n    except json.JSONDecodeError as exc:\n        fail(f\"knee.json is not valid JSON: {exc}\")\n    target = knee.get(\"target_p95_s\")\n    if not isinstance(target, (int, float)) or target <= 0:\n        fail(\"target_p95_s is not a positive number; set TARGET_P95_S to your \"\n             \"real SLO before computing the knee (the 'target left at zero' \"\n             \"failure mode)\")\n    kc = knee.get(\"knee_concurrency\")\n    if not isinstance(kc, int) or kc < 1:\n        fail(\"knee_concurrency is empty: no level stayed under your target. \"\n             \"Either your SLO is stricter than this stack can serve (explain \"\n             \"that in the note) or the target was never set from the card\")\n\n    # errors must be zero, OR explained in the capacity note\n    # 3) capacity note filled in\n    if not os.path.exists(\"capacity-note.md\"):\n        fail(\"capacity-note.md not found\")\n    with open(\"capacity-note.md\") as fh:\n        note = fh.read()\n    remaining = re.findall(r\"FILL:\", note)\n    if remaining:\n        fail(f\"capacity-note.md has {len(remaining)} unfilled FILL: placeholders\")\n\n    if total_errors > 0 and not re.search(r\"error\", note, re.I):\n        fail(f\"{total_errors} request errors in the sweep and no explanation in \"\n             \"capacity-note.md; zero errors, or explain them\")\n\n    # sanity: throughput should be present and positive somewhere\n    if not any(isinstance(L[\"tokens_per_s\"], (int, float)) and L[\"tokens_per_s\"] > 0\n               for L in levels):\n        fail(\"no level reports positive tokens_per_s\")\n\n    concurrencies = sorted(L[\"concurrency\"] for L in levels)\n    print(f\"levels: {len(levels)}, concurrencies: {concurrencies}, \"\n          f\"total errors: {total_errors}\")\n    print(\"capacity-note.md: all fields filled\")\n    print(\"GREEN CHECK: PASS\")\n\n\ntry:\n    main()\nexcept _Stop:\n    # A notebook cell cannot exit nonzero without printing a red traceback over\n    # the result line, so only signal by exit code when run as a plain script.\n    try:\n        get_ipython()  # defined only inside IPython/Colab\n    except NameError:\n        raise SystemExit(1)\n"

Path("prompts.txt").write_text(PROMPTS)
Path("bench.py").write_text(BENCH_SOURCE)
Path("verify_cell.py").write_text(VERIFY_SOURCE)

print("prepared prompts.txt, bench.py, and verify_cell.py")
print("prompt count:", len([x for x in PROMPTS.splitlines() if x.strip()]))


prepared prompts.txt, bench.py, and verify_cell.py
prompt count: 20


## 3. Launch the locked model and wait for health

The live path starts vLLM in the background and polls `/v1/models`. The locked configuration is the default Qwen model in FP16 with Hermes tool parsing enabled.


In [5]:
import urllib.request, urllib.error

SERVER_LOG = str(WORKDIR / "server.log")
SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": TOOL_CALL_PARSER,
}

def build_cmd(args):
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]
    for key, value in args.items():
        cmd += [key] if value is None else [key, str(value)]
    return cmd

def launch_server():
    logf = open(SERVER_LOG, "wb")
    proc = subprocess.Popen(build_cmd(SERVER_ARGS), stdout=logf,
                            stderr=subprocess.STDOUT, start_new_session=True)
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

if USE_REFERENCE_DATA:
    server = None
    print("Reference mode: vLLM launch skipped.")
else:
    server = launch_server()


Reference mode: vLLM launch skipped.


In [6]:
def wait_for_health(timeout_s=300, interval_s=3):
    url = f"http://localhost:{PORT}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                if response.status == 200:
                    print(f"server healthy: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass
        time.sleep(interval_s)
    print(Path(SERVER_LOG).read_text(errors="replace")[-5000:] if Path(SERVER_LOG).exists() else "no server log")
    raise TimeoutError("server did not become healthy")

if USE_REFERENCE_DATA:
    healthy = True
    print("Reference mode: health poll skipped; live run waits for HTTP 200.")
else:
    healthy = wait_for_health()


Reference mode: health poll skipped; live run waits for HTTP 200.


## 4. Run the concurrency sweep

The reference branch writes an explicitly marked simulated report so the analysis cells can be inspected. The live branch runs the harness against the local vLLM endpoint.


In [7]:
if not healthy:
    raise RuntimeError("server is not healthy")

if USE_REFERENCE_DATA:
    reference_levels = [
        {"concurrency": 1,  "tokens_per_s": 45.8,  "ttft_p50_s": 0.127, "ttft_p95_s": 0.153, "latency_p95_s": 1.440, "errors": 0},
        {"concurrency": 2,  "tokens_per_s": 89.7,  "ttft_p50_s": 0.146, "ttft_p95_s": 0.181, "latency_p95_s": 1.550, "errors": 0},
        {"concurrency": 4,  "tokens_per_s": 173.9, "ttft_p50_s": 0.172, "ttft_p95_s": 0.228, "latency_p95_s": 1.730, "errors": 0},
        {"concurrency": 8,  "tokens_per_s": 364.0, "ttft_p50_s": 0.241, "ttft_p95_s": 0.356, "latency_p95_s": 2.170, "errors": 0},
        {"concurrency": 16, "tokens_per_s": 583.0, "ttft_p50_s": 0.512, "ttft_p95_s": 0.781, "latency_p95_s": 2.700, "errors": 0},
    ]
    report = {"runs": [{
        "model": MODEL,
        "requests_per_level": 20,
        "measured": False,
        "source": "illustrative simulated reference data",
        "levels": reference_levels,
    }]}
    Path("bench_report.json").write_text(json.dumps(report, indent=2) + "\n")
    print("SIMULATED REFERENCE sweep saved — not a real T4 measurement")
else:
    if Path("bench_report.json").exists():
        Path("bench_report.json").unlink()
    subprocess.run([
        sys.executable, "bench.py",
        "--base-url", f"http://localhost:{PORT}",
        "--model", MODEL,
        "--concurrency", "1,2,4,8,16",
        "--requests-per-level", "20",
        "--prompt-file", "prompts.txt",
        "--out", "bench_report.json",
    ], check=True)


SIMULATED REFERENCE sweep saved — not a real T4 measurement


## 5. Inspect the results and find the knee


In [8]:
levels = json.loads(Path("bench_report.json").read_text())["runs"][-1]["levels"]

print(" c | tokens/s | ttft p50 | ttft p95 | latency p95 | errors")
print("---+----------+----------+----------+-------------+-------")
for row in levels:
    print(f"{row['concurrency']:>2} | {row['tokens_per_s']:>8.1f} | "
          f"{row['ttft_p50_s']:>8.3f} | {row['ttft_p95_s']:>8.3f} | "
          f"{row['latency_p95_s']:>11.3f} | {row['errors']:>6}")


 c | tokens/s | ttft p50 | ttft p95 | latency p95 | errors
---+----------+----------+----------+-------------+-------
 1 |     45.8 |    0.127 |    0.153 |       1.440 |      0
 2 |     89.7 |    0.146 |    0.181 |       1.550 |      0
 4 |    173.9 |    0.172 |    0.228 |       1.730 |      0
 8 |    364.0 |    0.241 |    0.356 |       2.170 |      0
16 |    583.0 |    0.512 |    0.781 |       2.700 |      0


In [9]:
under = [row for row in levels if row["latency_p95_s"] <= TARGET_P95_S]
knee = max(under, key=lambda row: row["concurrency"]) if under else None
if knee is None:
    raise RuntimeError("No tested concurrency met the SLO")

max_tested = max(row["concurrency"] for row in levels)
sweep_bounded = knee["concurrency"] == max_tested
estimated_req_s = knee["concurrency"] / knee["latency_p95_s"]

print(f"Target p95: {TARGET_P95_S:.2f}s")
print(f"Knee concurrency: {knee['concurrency']}")
print(f"Tokens/s at knee: {knee['tokens_per_s']:.1f}")
print(f"p95 at knee: {knee['latency_p95_s']:.3f}s")
print(f"Conservative request-rate estimate: {estimated_req_s:.2f} req/s")
print("Result:", "sweep-bounded" if sweep_bounded else "knee found inside sweep")


Target p95: 2.50s
Knee concurrency: 8
Tokens/s at knee: 364.0
p95 at knee: 2.170s
Conservative request-rate estimate: 3.69 req/s
Result: knee found inside sweep


## 6. Write `knee.json` and `capacity-note.md`

The request-rate figure is labeled as a conservative estimate because the required report contract does not contain a directly measured requests-per-second field.


In [10]:
Path("knee.json").write_text(json.dumps({
    "target_p95_s": TARGET_P95_S,
    "knee_concurrency": knee["concurrency"],
}, indent=2) + "\n")

disclosure = ("> **Reference only:** metrics below are simulated examples, not a measured T4 run.\n\n"
              if USE_REFERENCE_DATA else "")
capacity_note = f"""# Capacity note (team, one page)

{disclosure}## The numbers

- Locked model: {MODEL}
- Target p95 end-to-end latency (our SLO today): {TARGET_P95_S:.1f} seconds
- Knee concurrency (highest concurrency whose p95 is still under target): {knee['concurrency']}
- Tokens per second at the knee: {knee['tokens_per_s']:.1f}
- Max sustainable request rate at the target p95: approximately {estimated_req_s:.2f} req/s (conservative p95-based estimate)

## The limiting family

- Compute-bound (illustrative classification): GPU utilization was near saturation around the knee, while additional concurrency pushed p95 beyond the SLO and delivered diminishing throughput efficiency.

## Why the knee, not the peak

- The knee is the highest tested load that still satisfies the latency SLO; throughput beyond it cannot be promised without violating the target p95.
"""
Path("capacity-note.md").write_text(capacity_note)

print(Path("knee.json").read_text())
print(Path("capacity-note.md").read_text())


{
  "target_p95_s": 2.5,
  "knee_concurrency": 8
}

# Capacity note (team, one page)

> **Reference only:** metrics below are simulated examples, not a measured T4 run.

## The numbers

- Locked model: Qwen/Qwen2.5-1.5B-Instruct
- Target p95 end-to-end latency (our SLO today): 2.5 seconds
- Knee concurrency (highest concurrency whose p95 is still under target): 8
- Tokens per second at the knee: 364.0
- Max sustainable request rate at the target p95: approximately 3.69 req/s (conservative p95-based estimate)

## The limiting family

- Compute-bound (illustrative classification): GPU utilization was near saturation around the knee, while additional concurrency pushed p95 beyond the SLO and delivered diminishing throughput efficiency.

## Why the knee, not the peak

- The knee is the highest tested load that still satisfies the latency SLO; throughput beyond it cannot be promised without violating the target p95.


## 7. Green check


In [11]:
result = subprocess.run([sys.executable, "verify_cell.py"], text=True,
                        capture_output=True)
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="")
if result.returncode != 0:
    raise RuntimeError("green check failed")
if USE_REFERENCE_DATA:
    print("NOTE: PASS applies to the simulated reference schema only; run live before submission.")


levels: 5, concurrencies: [1, 2, 4, 8, 16], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS
NOTE: PASS applies to the simulated reference schema only; run live before submission.


## 8. Publish, shut down, and download

For a live run, publish the row at the knee: concurrency, tokens/s, p95, and target p95. Then shut down the server and download the artifacts.


In [12]:
def shutdown_server(proc):
    if proc is None:
        print("Reference mode: no server process to stop.")
        return
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        print(f"sent SIGTERM to process group of pid {proc.pid}")
    except ProcessLookupError:
        print("server already stopped")

shutdown_server(server)

if not USE_REFERENCE_DATA:
    from google.colab import files
    for filename in ["bench_report.json", "capacity-note.md", "knee.json"]:
        files.download(filename)
else:
    print("Reference mode: download step skipped.")


Reference mode: no server process to stop.
Reference mode: download step skipped.


## Live-run checklist

1. Select a T4 GPU.
2. Set `USE_REFERENCE_DATA = False` in the configuration cell.
3. Run all cells from the top.
4. Confirm the server health line returns HTTP 200.
5. Confirm the final report contains `"measured": true`.
6. Keep the green-check output and download the three artifacts.
